# Prototype with Oracle Select AI

**Time:** 30 minutes  
**Audience:** Application developers with general SQL and beginner-to-intermediate Python experience  
**Scenario:** Build an internal assistant for a fictional retailer's order-support team

By the end, you will be able to:

- map an OCI Generative AI or Ollama choice to a purpose-specific Select AI bundle;
- inspect generated SQL before it runs, then explain, run, and narrate the request;
- use curated feedback and a persistent conversation safely;
- generate a few fictional prototype rows;
- ask a grounded policy question with in-database ONNX embeddings;
- call a Select AI agent from a small `ipywidgets` application.

The LiveLabs setup has already created the database schema, credentials, profiles, ONNX model, vector index, policy corpus, and agent teams. You will not create infrastructure or handle provider secrets in this notebook.

> **First draft:** The notebook is statically validated. The exact pinned Oracle AI Database 26ai Free image and both provider paths still require the clean-room feasibility run described in the design specification.


## Route and time budget

| Time | Section | Outcome |
| ---: | --- | --- |
| 2 min | Welcome and preflight | Confirm that the lab is ready |
| 2 min | Provider bundle and metadata | Choose OCI or Ollama and inspect the data boundary |
| 4 min | NL2SQL | Generate and inspect SQL |
| 3 min | Explain, run, and narrate | Match the action to the application contract |
| 2 min | Feedback | Store or inspect one curated correction |
| 3 min | Conversation | Carry context through an explicit identifier |
| 2 min | Synthetic data | Create three disposable support cases |
| 3 min | RAG | Ground an answer in policy content |
| 9 min | Agent and Python prototype | Complete the order-support assistant |

```mermaid
flowchart LR
    U["You in JupyterLab 4.5.7"] -->|"python-oracledb"| D["Oracle AI Database 26ai Free"]
    D --> E["In-database ONNX embeddings"]
    D --> V["Retail data and policy vectors"]
    D --> O["Ollama on the private container network"]
    D --> G["OCI Generative AI"]
```

For NL2SQL generation, the provider receives your prompt and permitted schema metadata, not table rows. `NARRATE`, RAG, and agent calls can send query results or retrieved policy context to the selected provider. Generated text and SQL can vary by provider and run, so the checks below validate behavior instead of exact wording.


## 1. Configuration and reusable helpers

The notebook follows the existing lab convention: connection values come from `DBUSER`, `DBPASSWORD`, and `DBCONNECTION`. LiveLabs supplies them before participant time. No cell prints the password, database credential, private key, token, or full profile attributes.

The default provider is Ollama. Set `SELECTAI_PROVIDER=OCI` in the environment to make OCI the initial choice. The provider exercise later lets you change it interactively.


In [ ]:
from __future__ import annotations

import html
import json
import os
import re
from contextlib import contextmanager
from datetime import datetime, timezone
from typing import Any, Iterable, Iterator, Sequence

from IPython.display import HTML, Markdown, clear_output, display


def _status_error_text(exc: Exception) -> str:
    """Return a short error without exposing credentials or a stack trace."""
    message = " ".join(str(exc).split())
    message = re.sub(
        r"(?i)(password|token|secret|private[_ -]?key)\s*[:=]\s*[^\s,;]+",
        r"\1=[redacted]",
        message,
    )
    return message[:320] + ("..." if len(message) > 320 else "")


def ok(msg: str) -> None:
    """Print a green completion line at the bottom of a successful cell."""
    display(HTML(
        f"<span style='color:#1a7f37;font-weight:700'>&#10003;</span>"
        f" <span style='color:#1a7f37'>{html.escape(msg)}</span>"
    ))


def error(msg: str) -> None:
    """Print a red completion line at the bottom of a cell that encountered an error."""
    display(HTML(
        f"<span style='color:#cf222e;font-weight:700'>&#10007;</span>"
        f" <span style='color:#cf222e'>{html.escape(msg)}</span>"
    ))


@contextmanager
def cell_status(success_message: str) -> Iterator[None]:
    """Finish each cell with one green success or red error line."""
    try:
        yield
    except Exception as exc:
        error(
            "Cell encountered an error: "
            f"{type(exc).__name__}: {_status_error_text(exc)}"
        )
    else:
        ok(success_message)


with cell_status("Configuration loaded."):
    import ipywidgets as widgets
    import oracledb
    from dotenv import load_dotenv

    load_dotenv()

    DB_USER = os.getenv("DBUSER", "SELECTAI_LAB")
    DB_PASSWORD = os.getenv("DBPASSWORD", os.getenv("PASSWORD", "CHANGE_ME"))
    DB_DSN = os.getenv("DBCONNECTION", "aidbfree:1521/FREEPDB1")

    PROVIDER_KEY = os.getenv("SELECTAI_PROVIDER", "OLLAMA").strip().upper()
    FEEDBACK_MODE = os.getenv("SELECTAI_FEEDBACK_MODE", "READ_ONLY").strip().upper()
    ONNX_MODEL = os.getenv("SELECTAI_ONNX_MODEL", "ALL_MINILM_L12_V2").strip().upper()
    POLICY_VECTOR_INDEX = os.getenv("SELECTAI_POLICY_VECTOR_INDEX", "SELECTAI_POLICY_VECINDEX").strip().upper()

    PROVIDER_BUNDLES = {
        "OCI": {
            "label": "OCI Generative AI",
            "nl2sql_profile": "SELECTAI_OCI_NL2SQL",
            "rag_profile": "SELECTAI_OCI_RAG",
            "agent_team": "SELECTAI_OCI_TEAM",
            "model": os.getenv("SELECTAI_OCI_MODEL", "PINNED_BY_LIVELABS"),
            "embedding": f"In-database ONNX: {ONNX_MODEL}",
        },
        "OLLAMA": {
            "label": "Ollama",
            "nl2sql_profile": "SELECTAI_OLLAMA_NL2SQL",
            "rag_profile": "SELECTAI_OLLAMA_RAG",
            "agent_team": "SELECTAI_OLLAMA_TEAM",
            "model": os.getenv("SELECTAI_OLLAMA_MODEL", "llama3.2"),
            "embedding": f"In-database ONNX: {ONNX_MODEL}",
        },
    }

    ALLOWED_RETAIL_OBJECTS = {
        "CUSTOMERS", "PRODUCTS", "ORDERS", "ORDER_ITEMS",
        "SUPPORT_CASES", "POC_SUPPORT_CASES",
    }

    if PROVIDER_KEY not in PROVIDER_BUNDLES:
        PROVIDER_KEY = "OLLAMA"

    print(
        f"Configuration loaded for {DB_USER}@{DB_DSN}; "
        f"initial provider={PROVIDER_KEY}; feedback={FEEDBACK_MODE}."
    )


In [ ]:
with cell_status('Database connection and helper functions loaded.'):
    def read_lob(value: Any) -> Any:
        """Convert an Oracle LOB to a Python value without changing other values."""
        return value.read() if hasattr(value, "read") else value


    def short_error(exc: Exception) -> str:
        """Return one learner-facing line and avoid exposing a full stack trace."""
        message = " ".join(str(exc).split())
        message = re.sub(
            r"(?i)(password|token|secret|private[_ -]?key)\s*[:=]\s*[^\s,;]+",
            r"\1=[redacted]",
            message,
        )
        return message[:320] + ("..." if len(message) > 320 else "")


    def open_connection() -> oracledb.Connection:
        if not DB_PASSWORD or DB_PASSWORD == "CHANGE_ME":
            raise RuntimeError(
                "DBPASSWORD is not configured. Re-run the LiveLabs setup, then restart the kernel."
            )
        return oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)


    conn = open_connection()


    def query(
        sql: str,
        binds: dict[str, Any] | None = None,
        *,
        max_rows: int = 20,
    ) -> tuple[list[str], list[tuple[Any, ...]]]:
        with conn.cursor() as cursor:
            cursor.execute(sql, binds or {})
            columns = [item[0] for item in cursor.description] if cursor.description else []
            rows = cursor.fetchmany(max_rows) if cursor.description else []
        return columns, [tuple(read_lob(value) for value in row) for row in rows]


    def scalar(sql: str, binds: dict[str, Any] | None = None) -> Any:
        columns, rows = query(sql, binds, max_rows=1)
        return rows[0][0] if columns and rows else None


    def execute(sql: str, binds: dict[str, Any] | None = None) -> None:
        with conn.cursor() as cursor:
            cursor.execute(sql, binds or {})


    def show_table(columns: Sequence[str], rows: Iterable[Sequence[Any]], *, limit: int = 20) -> None:
        safe_rows = list(rows)[:limit]
        head = "".join(f"<th>{html.escape(str(name))}</th>" for name in columns)
        body = []
        for row in safe_rows:
            cells = "".join(
                f"<td><pre style='white-space:pre-wrap;margin:0'>{html.escape(str(value))}</pre></td>"
                for value in row
            )
            body.append(f"<tr>{cells}</tr>")
        display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"))


    def show_text(title: str, value: Any) -> None:
        text = str(read_lob(value) or "")
        display(Markdown(f"**{title}**"))
        display(HTML(f"<pre style='white-space:pre-wrap'>{html.escape(text)}</pre>"))


    def call_select_ai(
        prompt: str,
        profile_name: str,
        action: str,
        *,
        conversation_id: str | None = None,
    ) -> str:
        params = json.dumps({"conversation_id": conversation_id}) if conversation_id else None
        result = scalar(
            """
            SELECT DBMS_CLOUD_AI.GENERATE(
                     prompt       => :prompt,
                     profile_name => :profile_name,
                     action       => :action,
                     params       => :params
                   )
            FROM dual
            """,
            {
                "prompt": prompt,
                "profile_name": profile_name,
                "action": action.upper(),
                "params": params,
            },
        )
        return str(read_lob(result) or "")


    def create_conversation(title: str) -> str:
        attributes = json.dumps(
            {
                "title": title,
                "description": "Select AI LiveLabs order-support prototype",
                "retention_days": 1,
                "conversation_length": 4,
            }
        )
        return str(
            scalar(
                "SELECT DBMS_CLOUD_AI.CREATE_CONVERSATION(attributes => :attributes) FROM dual",
                {"attributes": attributes},
            )
        )


    def run_team(team_name: str, prompt: str, conversation_id: str) -> str:
        params = json.dumps({"conversation_id": conversation_id})
        result = scalar(
            """
            SELECT DBMS_CLOUD_AI_AGENT.RUN_TEAM(
                     team_name   => :team_name,
                     user_prompt => :user_prompt,
                     params      => :params
                   )
            FROM dual
            """,
            {"team_name": team_name, "user_prompt": prompt, "params": params},
        )
        return str(read_lob(result) or "")


    print("Database connection and Select AI helpers are ready.")


## 2. Read-only preflight (2 minutes)

This cell checks the database release, packages, lab objects, profiles, ONNX model, vector index, agent teams, and the initially selected provider. It shows only names, statuses, and short diagnostics. A required failure stops the lab early so you do not spend time debugging later exercises.


In [ ]:
with cell_status('Preflight completed successfully.'):
    checks: list[tuple[str, str, str]] = []


    def record_check(label: str, passed: bool, detail: str, *, required: bool = True) -> None:
        state = "PASS" if passed else ("FAIL" if required else "WARN")
        checks.append((state, label, detail))


    try:
        current_user, current_pdb = query(
            "SELECT SYS_CONTEXT('USERENV','CURRENT_USER'), SYS_CONTEXT('USERENV','CON_NAME') FROM dual",
            max_rows=1,
        )[1][0]
        record_check(
            "Database connection",
            current_user == DB_USER.upper() and bool(current_pdb),
            f"user={current_user}; pdb={current_pdb}",
        )
    except Exception as exc:
        record_check("Database connection", False, short_error(exc))

    try:
        version_text = str(
            scalar(
                """
                SELECT version_full
                FROM product_component_version
                WHERE product LIKE 'Oracle Database%'
                FETCH FIRST 1 ROW ONLY
                """
            )
            or ""
        )
        version_numbers = tuple(int(item) for item in re.findall(r"\d+", version_text)[:3])
        record_check(
            "Oracle AI Database release",
            version_numbers >= (23, 26, 1),
            f"reported={version_text or 'unknown'}; target RU=23.26.1 or later",
        )
    except Exception as exc:
        record_check("Oracle AI Database release", False, short_error(exc))

    required_members = {
        "DBMS_CLOUD_AI": {
            "GENERATE", "FEEDBACK", "CREATE_CONVERSATION",
            "GENERATE_SYNTHETIC_DATA", "CREATE_VECTOR_INDEX", "SET_CONVERSATION_ID",
        },
        "DBMS_CLOUD_AI_AGENT": {"RUN_TEAM", "DESCRIBE_TEAM"},
    }
    for package_name, members in required_members.items():
        try:
            columns, rows = query(
                """
                SELECT DISTINCT procedure_name
                FROM all_procedures
                WHERE object_name = :package_name
                  AND procedure_name IS NOT NULL
                """,
                {"package_name": package_name},
                max_rows=200,
            )
            available = {str(row[0]).upper() for row in rows}
            missing = sorted(members - available)
            record_check(
                package_name,
                not missing,
                "required members visible" if not missing else f"missing: {', '.join(missing)}",
            )
        except Exception as exc:
            record_check(package_name, False, short_error(exc))

    try:
        pipeline_count = int(
            scalar(
                "SELECT COUNT(*) FROM all_objects WHERE object_name='DBMS_CLOUD_PIPELINE' AND object_type='PACKAGE'"
            )
            or 0
        )
        record_check(
            "DBMS_CLOUD_PIPELINE",
            pipeline_count > 0,
            "available" if pipeline_count else "not visible; required only by provisioning",
            required=False,
        )
    except Exception as exc:
        record_check("DBMS_CLOUD_PIPELINE", False, short_error(exc), required=False)

    for feature_label, owner_name, package_name in (
        ("Oracle Text", "CTXSYS", "CTX_DDL"),
        ("AI Vector Search", "SYS", "DBMS_VECTOR"),
    ):
        try:
            feature_count = int(
                scalar(
                    """
                    SELECT COUNT(*)
                    FROM all_objects
                    WHERE owner = :owner_name
                      AND object_name = :package_name
                      AND object_type = 'PACKAGE'
                    """,
                    {"owner_name": owner_name, "package_name": package_name},
                )
                or 0
            )
            record_check(feature_label, feature_count > 0, f"package={owner_name}.{package_name}")
        except Exception as exc:
            record_check(feature_label, False, short_error(exc))

    try:
        object_rows = query(
            """
            SELECT table_name
            FROM user_tables
            WHERE table_name IN (
              'CUSTOMERS','PRODUCTS','ORDERS','ORDER_ITEMS',
              'SUPPORT_CASES','POC_SUPPORT_CASES'
            )
            """,
            max_rows=20,
        )[1]
        present_objects = {str(row[0]) for row in object_rows}
        missing_objects = sorted(ALLOWED_RETAIL_OBJECTS - present_objects)
        record_check(
            "Retail schema objects",
            not missing_objects,
            "all six tables present" if not missing_objects else f"missing: {', '.join(missing_objects)}",
        )
    except Exception as exc:
        record_check("Retail schema objects", False, short_error(exc))

    required_profiles = {
        item[key]
        for item in PROVIDER_BUNDLES.values()
        for key in ("nl2sql_profile", "rag_profile")
    }
    try:
        profile_rows = query(
            "SELECT profile_name, status FROM user_cloud_ai_profiles",
            max_rows=50,
        )[1]
        profile_status = {str(name): str(status) for name, status in profile_rows}
        missing_profiles = sorted(required_profiles - set(profile_status))
        disabled_profiles = sorted(
            name for name in required_profiles
            if name in profile_status and profile_status[name].upper() != "ENABLED"
        )
        record_check(
            "OCI and Ollama profile bundles",
            not missing_profiles and not disabled_profiles,
            f"missing={missing_profiles or 'none'}; disabled={disabled_profiles or 'none'}",
        )
    except Exception as exc:
        record_check("OCI and Ollama profile bundles", False, short_error(exc))

    try:
        model_count = int(
            scalar(
                "SELECT COUNT(*) FROM user_mining_models WHERE model_name = :model_name",
                {"model_name": ONNX_MODEL},
            )
            or 0
        )
        record_check("In-database ONNX model", model_count == 1, f"model={ONNX_MODEL}")
    except Exception as exc:
        record_check("In-database ONNX model", False, short_error(exc))

    try:
        index_name_column = scalar(
            """
            SELECT column_name
            FROM user_tab_columns
            WHERE table_name = 'USER_CLOUD_VECTOR_INDEXES'
              AND column_name IN ('INDEX_NAME', 'VECTOR_INDEX_NAME')
            ORDER BY CASE column_name WHEN 'INDEX_NAME' THEN 1 ELSE 2 END
            FETCH FIRST 1 ROW ONLY
            """
        )
        if index_name_column not in {"INDEX_NAME", "VECTOR_INDEX_NAME"}:
            raise RuntimeError("Vector index metadata view does not expose a recognized name column.")
        index_count = int(
            scalar(
                f"SELECT COUNT(*) FROM user_cloud_vector_indexes WHERE {index_name_column} = :index_name",
                {"index_name": POLICY_VECTOR_INDEX},
            )
            or 0
        )
        record_check("Policy vector index", index_count == 1, f"index={POLICY_VECTOR_INDEX}")
    except Exception as exc:
        record_check("Policy vector index", False, short_error(exc))

    try:
        team_json = str(read_lob(scalar("SELECT DBMS_CLOUD_AI_AGENT.LIST_TEAMS() FROM dual")) or "")
        missing_teams = [
            item["agent_team"] for item in PROVIDER_BUNDLES.values()
            if item["agent_team"].upper() not in team_json.upper()
        ]
        record_check(
            "OCI and Ollama agent teams",
            not missing_teams,
            "both teams listed" if not missing_teams else f"missing: {', '.join(missing_teams)}",
        )
        for configured_team in (item["agent_team"] for item in PROVIDER_BUNDLES.values()):
            description = str(
                read_lob(
                    scalar(
                        "SELECT DBMS_CLOUD_AI_AGENT.DESCRIBE_TEAM(:team_name) FROM dual",
                        {"team_name": configured_team},
                    )
                )
                or ""
            )
            record_check(
                f"Agent objects for {configured_team}",
                bool(description.strip()),
                "team description and configured skills are visible",
            )
    except Exception as exc:
        record_check("OCI and Ollama agent teams", False, short_error(exc))

    try:
        initial_bundle = PROVIDER_BUNDLES[PROVIDER_KEY]
        reachability_sql = call_select_ai(
            "Count the customers in the retail schema.",
            initial_bundle["nl2sql_profile"],
            "SHOWSQL",
        )
        record_check(
            "Selected provider reachability",
            bool(reachability_sql.strip()),
            f"provider={initial_bundle['label']}; profile={initial_bundle['nl2sql_profile']}",
        )
    except Exception as exc:
        record_check("Selected provider reachability", False, short_error(exc))

    record_check(
        "Feedback isolation",
        FEEDBACK_MODE == "WRITE",
        "participant-owned profile confirmed" if FEEDBACK_MODE == "WRITE" else "read-only demonstration mode",
        required=False,
    )

    show_table(["STATE", "CHECK", "DETAIL"], checks)

    required_failures = [label for state, label, _ in checks if state == "FAIL"]
    if required_failures:
        raise RuntimeError(
            "Preflight failed: " + ", ".join(required_failures) +
            ". Re-run LiveLabs provisioning before continuing."
        )

    print("Preflight passed. Continue with the provider exercise.")


## 3. Choose a provider bundle and inspect the data boundary (2 minutes)

### Exercise 1: Provider selection

**Objective:** Map one provider key to its NL2SQL profile, RAG profile, and agent team.

**Directions:** Change `provider_key` to `"OCI"` or `"OLLAMA"`. Run this cell, then run the hidden solution immediately below it. If a solution cell appears collapsed, click its left gutter or the collapsed-cell indicator to reveal it. In JupyterLab, the Cell Inspector also shows the `solution` tag.

**Expected result:** A bundle with two profiles, one team, one generative model, and in-database ONNX embeddings.


In [ ]:
with cell_status('Provider choice recorded.'):
    # TODO: Choose "OCI" or "OLLAMA".
    provider_key = PROVIDER_KEY

    # Hint: Provider-specific branching should end at this mapping.
    print(f"Selected key: {provider_key}")


In [ ]:
with cell_status('Provider bundle selected.'):
    provider_key = provider_key.strip().upper()
    if provider_key not in PROVIDER_BUNDLES:
        raise ValueError("Choose OCI or OLLAMA.")

    PROVIDER_KEY = provider_key
    bundle = PROVIDER_BUNDLES[PROVIDER_KEY]

    show_table(
        ["SETTING", "VALUE"],
        [
            ("Provider", bundle["label"]),
            ("NL2SQL profile", bundle["nl2sql_profile"]),
            ("RAG profile", bundle["rag_profile"]),
            ("Agent team", bundle["agent_team"]),
            ("Generative model", bundle["model"]),
            ("Embedding", bundle["embedding"]),
            ("Policy vector index", POLICY_VECTOR_INDEX),
        ],
    )


In [ ]:
with cell_status('Provider bundle validation completed.'):
    # Deterministic validation: the bundle must be complete and use distinct profiles.
    required_bundle_keys = {
        "label", "nl2sql_profile", "rag_profile", "agent_team", "model", "embedding"
    }
    bundle_ok = (
        required_bundle_keys <= set(bundle)
        and bundle["nl2sql_profile"] != bundle["rag_profile"]
        and ONNX_MODEL in bundle["embedding"]
    )
    if not bundle_ok:
        raise RuntimeError("Provider bundle is incomplete.")
    print("PASS: provider bundle is complete.")


In [ ]:
with cell_status('Profile attributes inspected.'):
    SAFE_PROFILE_ATTRIBUTES = {
        "provider", "model", "comments", "annotations", "constraints",
        "object_list", "enforce_object_list", "vector_index_name", "embedding_model",
    }


    def sanitized_profile_attributes(profile_name: str) -> dict[str, Any]:
        """Read only approved profile fields and never display credentials or tokens."""
        try:
            raw = scalar(
                "SELECT attributes FROM user_cloud_ai_profiles WHERE profile_name = :profile_name",
                {"profile_name": profile_name},
            )
            parsed = json.loads(str(read_lob(raw) or "{}"))
            return {key: parsed.get(key) for key in sorted(SAFE_PROFILE_ATTRIBUTES) if key in parsed}
        except Exception as exc:
            return {"inspection_note": short_error(exc)}


    nl2sql_safe = sanitized_profile_attributes(bundle["nl2sql_profile"])
    rag_safe = sanitized_profile_attributes(bundle["rag_profile"])
    show_table(
        ["PROFILE", "SANITIZED ATTRIBUTES"],
        [
            (bundle["nl2sql_profile"], json.dumps(nl2sql_safe, indent=2, default=str)),
            (bundle["rag_profile"], json.dumps(rag_safe, indent=2, default=str)),
        ],
    )


The profile's `object_list` is an AI-generation boundary. Database privileges remain the execution boundary. Both matter: a generated query should reference only approved objects, and the connected user should have only the privileges the application needs.

Before asking the model, inspect the metadata that gives business meaning and a declared join path.


In [ ]:
with cell_status('Schema metadata inspected.'):
    metadata_rows: list[tuple[str, str, str]] = []

    table_comment = scalar(
        "SELECT comments FROM user_tab_comments WHERE table_name='ORDERS'"
    )
    metadata_rows.append(("Table comment", "ORDERS", str(table_comment or "Not provisioned")))

    for column_name in ("STATUS", "DELIVERED_DATE"):
        comment = scalar(
            """
            SELECT comments
            FROM user_col_comments
            WHERE table_name='ORDERS' AND column_name=:column_name
            """,
            {"column_name": column_name},
        )
        metadata_rows.append(("Column comment", f"ORDERS.{column_name}", str(comment or "Not provisioned")))

    fk_columns, fk_rows = query(
        """
        SELECT c.constraint_name,
               cc.table_name || '.' || cc.column_name AS child_column,
               p.table_name AS parent_table
        FROM user_constraints c
        JOIN user_cons_columns cc ON cc.constraint_name = c.constraint_name
        JOIN user_constraints p ON p.constraint_name = c.r_constraint_name
        WHERE c.constraint_type='R'
          AND cc.table_name='ORDERS'
        FETCH FIRST 1 ROW ONLY
        """
    )
    if fk_rows:
        metadata_rows.append(("Foreign key", str(fk_rows[0][1]), str(fk_rows[0][2])))

    show_table(["TYPE", "OBJECT", "MEANING OR TARGET"], metadata_rows)


## 4. Natural language to SQL (4 minutes)

### Exercise 2: Generate SQL without running it

**Objective:** Use `SHOWSQL` for a multi-table retail question.

**Directions:** Edit the prompt so it asks for a small, useful result involving customers and orders. Keep `action = "SHOWSQL"` so the first call is inspectable and side-effect free. Then run the hidden solution.

**Expected result:** One read-only query that joins approved retail objects and includes a small result limit.


In [ ]:
with cell_status('NL2SQL prompt prepared.'):
    # TODO: Adapt this natural-language request, but keep it narrow enough for the local model.
    nl2sql_prompt = (
        "Which three customers have the most delayed orders in the last 30 days? "
        "Include customer name, region, and delayed-order count."
    )
    action = "SHOWSQL"

    print(nl2sql_prompt)


In [ ]:
with cell_status('SHOWSQL generation completed.'):
    if action.upper() != "SHOWSQL":
        raise ValueError("Use SHOWSQL before any action that executes or narrates a query.")

    generated_sql = call_select_ai(
        nl2sql_prompt,
        bundle["nl2sql_profile"],
        action,
    )
    show_text("Generated SQL", generated_sql)


In [ ]:
with cell_status('Generated SQL validation completed.'):
    def validate_generated_sql(sql_text: str) -> tuple[bool, list[str]]:
        normalized = re.sub(r"```(?:sql)?|```", "", sql_text, flags=re.IGNORECASE).strip()
        violations: list[str] = []
        if not re.search(r"\b(SELECT|WITH)\b", normalized, flags=re.IGNORECASE):
            violations.append("No SELECT or WITH statement found")
        if re.search(r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|TRUNCATE|GRANT)\b", normalized, flags=re.IGNORECASE):
            violations.append("A modifying statement was found")
        known_table_tokens = {
            token.upper() for token in re.findall(r"\b(?:FROM|JOIN)\s+\"?([A-Za-z][\w$#]*)\"?", normalized, flags=re.IGNORECASE)
        }
        forbidden = sorted(known_table_tokens - ALLOWED_RETAIL_OBJECTS - {"DUAL"})
        if forbidden:
            violations.append("Objects outside the allowlist: " + ", ".join(forbidden))
        return not violations, violations


    sql_ok, sql_violations = validate_generated_sql(generated_sql)
    print("PASS: generated SQL is read-only and scoped." if sql_ok else "REVIEW: " + "; ".join(sql_violations))


## 5. Explain, run, and narrate (3 minutes)

The four actions have different application contracts:

| Action | Use it when you need |
| --- | --- |
| `SHOWSQL` | SQL text for review, testing, or a diagnostic panel |
| `EXPLAINSQL` | A plain-language explanation of the generated query |
| `RUNSQL` | The query result for structured application behavior |
| `NARRATE` | User-facing prose based on the query result |

Structured application logic should consume rows or structured output. Do not parse a narrative to make business decisions.


In [ ]:
with cell_status('Select AI action comparison completed.'):
    action_outputs: list[tuple[str, str]] = [("SHOWSQL", generated_sql)]

    for next_action in ("EXPLAINSQL", "RUNSQL", "NARRATE"):
        value = call_select_ai(
            nl2sql_prompt,
            bundle["nl2sql_profile"],
            next_action,
        )
        action_outputs.append((next_action, value))

    show_table(["ACTION", "OUTPUT"], action_outputs, limit=4)


## 6. Curate Select AI Feedback (2 minutes)

Feedback stores examples that can be retrieved as hints for similar prompts. It does not fine-tune the LLM.

Only a profile owner or trusted curator should change shared behavior. This notebook writes feedback only when LiveLabs sets `SELECTAI_FEEDBACK_MODE=WRITE` for a participant-owned profile. Otherwise, it performs a read-only demonstration.


In [ ]:
with cell_status('Initial Feedback SQL generated.'):
    ambiguous_prompt = "Show the open support cases and their order numbers."
    initial_open_case_sql = call_select_ai(
        ambiguous_prompt,
        bundle["nl2sql_profile"],
        "SHOWSQL",
    )
    show_text("SQL before curated feedback", initial_open_case_sql)


### Exercise 3: Define one trusted correction

**Objective:** Teach the profile that an open case has status `NEW`, `IN_PROGRESS`, or `WAITING_CUSTOMER`.

**Directions:** Review the correction text and expected SQL. In an isolated environment, run the hidden solution to store and verify it. In read-only mode, the same cell shows what would be stored without changing the profile.

**Expected result:** The correction is either verified in the profile-specific feedback vector table or clearly reported as read-only.


In [ ]:
with cell_status('Feedback correction prepared.'):
    # TODO: Curate the business definition and review the expected SQL.
    feedback_content = (
        "An open support case has status NEW, IN_PROGRESS, or WAITING_CUSTOMER. "
        "Exclude RESOLVED and CLOSED cases."
    )
    corrected_sql = (
        "SELECT case_id, order_id, status, summary "
        "FROM support_cases "
        "WHERE status IN ('NEW','IN_PROGRESS','WAITING_CUSTOMER')"
    )

    print(feedback_content)


In [ ]:
with cell_status('Feedback exercise completed.'):
    feedback_profile = bundle["nl2sql_profile"]
    feedback_sql_text = f"select ai showsql {ambiguous_prompt}"

    if feedback_profile not in required_profiles:
        raise RuntimeError("Feedback profile is not in the reviewed provider mapping.")

    feedback_table = f"{feedback_profile}_FEEDBACK_VECINDEX$VECTAB"
    feedback_present = False
    try:
        feedback_present = int(
            scalar(
                f"SELECT COUNT(*) FROM {feedback_table} WHERE content = :prompt",
                {"prompt": ambiguous_prompt},
            )
            or 0
        ) > 0
    except oracledb.DatabaseError:
        feedback_present = False

    if FEEDBACK_MODE == "WRITE" and not feedback_present:
        execute(
            """
            BEGIN
              DBMS_CLOUD_AI.FEEDBACK(
                profile_name     => :profile_name,
                sql_text         => :sql_text,
                feedback_type    => 'negative',
                response         => :response,
                feedback_content => :feedback_content,
                operation        => 'add'
              );
            END;
            """,
            {
                "profile_name": feedback_profile,
                "sql_text": feedback_sql_text,
                "response": corrected_sql,
                "feedback_content": feedback_content,
            },
        )
        conn.commit()
        feedback_present = True
    elif FEEDBACK_MODE != "WRITE":
        print("READ_ONLY: no profile mutation was attempted.")

    if feedback_present:
        columns, rows = query(
            f"""
            SELECT content,
                   JSON_VALUE(attributes, '$.feedback_type') AS feedback_type,
                   JSON_VALUE(attributes, '$.response') AS corrected_response
            FROM {feedback_table}
            WHERE content = :prompt
            FETCH FIRST 1 ROW ONLY
            """,
            {"prompt": ambiguous_prompt},
            max_rows=1,
        )
        show_table(columns, rows)
    else:
        show_table(
            ["MODE", "PROFILE", "CORRECTION READY"],
            [("READ_ONLY", feedback_profile, corrected_sql)],
        )


In [ ]:
with cell_status('Post-feedback SQL validation completed.'):
    similar_prompt = "List open support cases with the related customer and order."
    post_feedback_sql = call_select_ai(
        similar_prompt,
        bundle["nl2sql_profile"],
        "SHOWSQL",
    )
    post_feedback_ok, post_feedback_issues = validate_generated_sql(post_feedback_sql)
    show_text("SQL for a similar prompt", post_feedback_sql)
    print(
        "PASS: similar SQL stayed within the permitted object boundary."
        if post_feedback_ok
        else "REVIEW: " + "; ".join(post_feedback_issues)
    )


## 7. Keep context with a persistent conversation (3 minutes)

### Exercise 4: Create a two-turn conversation

**Objective:** Keep a conversation identifier as application state instead of relying on one pooled database session.

**Directions:** Edit the follow-up so it depends on the first request. The solution creates a named conversation with one-day retention, then passes its identifier explicitly to both `DBMS_CLOUD_AI.GENERATE` calls.

**Expected result:** The follow-up SQL preserves the delayed-order topic and adds customer names.


In [ ]:
with cell_status('Conversation prompts prepared.'):
    # TODO: Make the second prompt depend on the first prompt.
    conversation_title = f"select-ai-lab-{datetime.now(timezone.utc):%Y%m%d}"
    first_prompt = "Show delayed orders by customer region."
    follow_up_prompt = "Show only the top three and include the customer name."

    print(first_prompt)
    print(follow_up_prompt)


In [ ]:
with cell_status('Two-turn conversation completed.'):
    if "conversation_id" not in globals() or not conversation_id:
        conversation_id = create_conversation(conversation_title)

    first_turn_sql = call_select_ai(
        first_prompt,
        bundle["nl2sql_profile"],
        "SHOWSQL",
        conversation_id=conversation_id,
    )
    follow_up_sql = call_select_ai(
        follow_up_prompt,
        bundle["nl2sql_profile"],
        "SHOWSQL",
        conversation_id=conversation_id,
    )

    show_table(
        ["CONVERSATION", "TURN", "GENERATED SQL"],
        [
            (conversation_id, "Initial", first_turn_sql),
            (conversation_id, "Follow-up", follow_up_sql),
        ],
    )


In [ ]:
with cell_status('Conversation validation completed.'):
    conversation_ok = bool(conversation_id and first_turn_sql.strip() and follow_up_sql.strip())
    if not conversation_ok:
        raise RuntimeError("Conversation state is incomplete.")
    print("PASS: two turns used one explicit conversation ID.")

    try:
        audit_columns, audit_rows = query(
            """
            SELECT *
            FROM user_cloud_ai_conversation_prompts
            WHERE conversation_id = :conversation_id
            FETCH FIRST 1 ROW ONLY
            """,
            {"conversation_id": conversation_id},
            max_rows=1,
        )
        safe_audit_names = {
            "CONVERSATION_ID", "PROMPT_ID", "PROFILE_NAME", "ACTION",
            "CREATED_AT", "CREATED_ON", "CREATED_TIMESTAMP",
        }
        safe_positions = [
            index for index, name in enumerate(audit_columns)
            if name.upper() in safe_audit_names
        ]
        safe_columns = [audit_columns[index] for index in safe_positions]
        safe_rows = [tuple(row[index] for index in safe_positions) for row in audit_rows]
        show_table(safe_columns or ["AUDIT"], safe_rows or [("Conversation row stored",)], limit=1)
    except Exception as exc:
        print("Audit row unavailable: " + short_error(exc))


## 8. Generate disposable prototype data (2 minutes)

### Exercise 5: Create three fictional support cases

**Objective:** Populate only `POC_SUPPORT_CASES`, the lab's resettable sandbox.

**Directions:** Keep the record count low. Adjust the guidance if you want a different mix of categories. The solution clears only the disposable table, uses existing fictional order/customer pairs, generates rows, and commits them.

**Expected result:** Three rows whose statuses satisfy the table constraint.


In [ ]:
with cell_status('Synthetic-data request prepared.'):
    # TODO: Adjust the low record count or generation guidance.
    synthetic_record_count = 3
    synthetic_guidance = (
        "Generate fictional retail support cases. Use only RETURN, REFUND, SHIPPING, "
        "or WARRANTY categories and only NEW, IN_PROGRESS, or WAITING_CUSTOMER statuses."
    )

    print(f"Requested rows: {synthetic_record_count}")


In [ ]:
with cell_status('Synthetic data generation completed.'):
    if not 1 <= synthetic_record_count <= 5:
        raise ValueError("Use between 1 and 5 synthetic rows for this timed lab.")

    pair_columns, pair_rows = query(
        "SELECT customer_id, order_id FROM orders FETCH FIRST 3 ROWS ONLY",
        max_rows=3,
    )
    pair_text = ", ".join(f"customer {customer_id} with order {order_id}" for customer_id, order_id in pair_rows)
    generation_prompt = synthetic_guidance + f" Use these valid pairs when needed: {pair_text}."

    execute("DELETE FROM poc_support_cases")
    execute(
        """
        BEGIN
          DBMS_CLOUD_AI.GENERATE_SYNTHETIC_DATA(
            profile_name => :profile_name,
            object_name  => 'POC_SUPPORT_CASES',
            owner_name   => :owner_name,
            record_count => :record_count,
            user_prompt  => :user_prompt,
            params       => '{"sample_rows":0,"table_statistics":false}'
          );
        END;
        """,
        {
            "profile_name": bundle["nl2sql_profile"],
            "owner_name": DB_USER.upper(),
            "record_count": synthetic_record_count,
            "user_prompt": generation_prompt,
        },
    )
    conn.commit()

    synthetic_columns, synthetic_rows = query(
        "SELECT * FROM poc_support_cases FETCH FIRST 5 ROWS ONLY",
        max_rows=5,
    )
    show_table(synthetic_columns, synthetic_rows)


In [ ]:
with cell_status('Synthetic data validation completed.'):
    generated_count = int(scalar("SELECT COUNT(*) FROM poc_support_cases") or 0)
    invalid_status_count = int(
        scalar(
            """
            SELECT COUNT(*)
            FROM poc_support_cases
            WHERE status NOT IN ('NEW','IN_PROGRESS','WAITING_CUSTOMER','RESOLVED','CLOSED')
               OR status IS NULL
            """
        )
        or 0
    )
    synthetic_ok = generated_count == synthetic_record_count and invalid_status_count == 0
    if not synthetic_ok:
        raise RuntimeError(
            f"Synthetic-data validation failed: rows={generated_count}; "
            f"invalid statuses={invalid_status_count}."
        )
    print(f"PASS: {generated_count} constrained fictional rows generated.")
    show_table(synthetic_columns, synthetic_rows)


Generated data is useful for prototypes and tests, not as production truth. This exercise uses a disposable table so it cannot alter the deterministic order facts used by the remaining sections.

## 9. Ask a grounded policy question with RAG (3 minutes)

The selected RAG profile embeds the question with the imported ONNX model inside Oracle AI Database, searches the prebuilt policy vector index, and sends retrieved context to the selected generative provider.

### Exercise 6: Request a grounded policy answer

**Objective:** Ask a narrow return-policy question and require source metadata in the answer.

**Directions:** Change the question while keeping the instruction to cite the policy document ID and title.

**Expected result:** A concise answer grounded in the fictional policy corpus, not general model knowledge.


In [ ]:
with cell_status('RAG question prepared.'):
    # TODO: Adapt the policy question, but retain the grounding and source requirement.
    rag_question = (
        "Using only the indexed retail policy documents, explain the return window for a delivered item. "
        "Include the policy document ID and title used as the source."
    )

    print(rag_question)


In [ ]:
with cell_status('RAG request completed.'):
    rag_answer = call_select_ai(
        rag_question,
        bundle["rag_profile"],
        "CHAT",
    )
    show_text("Grounded policy answer", rag_answer)
    show_table(
        ["RETRIEVAL SETTING", "VALUE"],
        [
            ("RAG profile", bundle["rag_profile"]),
            ("Embedding model", ONNX_MODEL),
            ("Vector index", POLICY_VECTOR_INDEX),
        ],
    )


In [ ]:
with cell_status('RAG validation completed.'):
    rag_lower = rag_answer.lower()
    rag_ok = bool(rag_answer.strip()) and "return" in rag_lower and any(
        marker in rag_lower for marker in ("source", "policy", "document")
    )
    print(
        "PASS: the answer is non-empty and exposes policy/source language."
        if rag_ok
        else "REVIEW: confirm that the RAG profile returns source metadata for this corpus."
    )
    show_text("Validated grounded answer", rag_answer)


## 10. Combine order facts and policy guidance with an agent (part of the final 9 minutes)

The agent team is preprovisioned because object creation is infrastructure work. Its NL2SQL tool uses the selected bundle's NL2SQL profile, and its RAG tool uses the matching RAG profile. A read-only `GET_ALLOWED_SUPPORT_ACTIONS` tool may be present when it passes the two-provider timing gate.


In [ ]:
with cell_status('Agent team inspected.'):
    team_description = scalar(
        "SELECT DBMS_CLOUD_AI_AGENT.DESCRIBE_TEAM(:team_name) FROM dual",
        {"team_name": bundle["agent_team"]},
    )
    show_text("Sanitized team description", team_description)


### Exercise 7: Run one combined order-and-policy task

**Objective:** Ask the agent to use structured order data and policy content in one response.

**Directions:** Keep order `1042` for the deterministic seed case, or replace it with another order shown by the final selector. Require the response to separate facts, policy guidance, and the next support action.

**Expected result:** One response with order facts, applicable policy, source information, and a recommended next action.


In [ ]:
with cell_status('Agent request prepared.'):
    # TODO: Adapt the support request while retaining its required answer structure.
    agent_prompt = (
        "Review order 1042. Separate verified order facts from policy guidance, "
        "cite the policy source, and identify the next permitted support action."
    )

    print(agent_prompt)


In [ ]:
with cell_status('Agent workflow completed.'):
    if "conversation_id" not in globals() or not conversation_id:
        conversation_id = create_conversation("select-ai-agent-lab")

    agent_answer = run_team(
        bundle["agent_team"],
        agent_prompt,
        conversation_id,
    )
    show_text("Agent response", agent_answer)


In [ ]:
with cell_status('Agent response validation completed.'):
    agent_lower = agent_answer.lower()
    agent_checks = {
        "order reference": "1042" in agent_lower,
        "policy guidance": "policy" in agent_lower,
        "support action": any(word in agent_lower for word in ("action", "next", "contact", "return", "refund")),
    }
    show_table(
        ["BEHAVIORAL CHECK", "RESULT"],
        [(name, "PASS" if passed else "REVIEW") for name, passed in agent_checks.items()],
    )
    show_text("Validated agent answer", agent_answer)


## 11. Complete the Python order-support prototype (remaining final 9 minutes)

This small interface keeps the application path visible: validate inputs, map the provider bundle, create or reuse a conversation ID, call the database agent team, convert LOBs, and present a short recovery message if a provider or database call fails.

### Exercise 8: Complete the button handler

**Objective:** Connect an `ipywidgets` event to `DBMS_CLOUD_AI_AGENT.RUN_TEAM`.

**Directions:** Read the starter handler. Either replace its placeholder with the steps in the comments or run the hidden complete solution immediately below it.

**Expected result:** An interface with provider, order, question, answer, and diagnostic areas.


In [ ]:
with cell_status('Draft widget handler loaded.'):
    # TODO: Replace the placeholder message with input validation, provider mapping,
    # conversation state, run_team(...), and short error handling.
    def handle_support_request_draft(provider_key: str, order_id: int, question: str) -> str:
        if not question.strip():
            return "Enter a support question."
        return "TODO: call the selected Select AI agent team."


    print(handle_support_request_draft(PROVIDER_KEY, 1042, "Can the customer return this item?"))


In [ ]:
with cell_status('Complete widget handler loaded.'):
    order_columns, order_rows = query(
        """
        SELECT o.order_id,
               c.full_name || ' | ' || o.status AS label
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        ORDER BY o.order_id
        FETCH FIRST 20 ROWS ONLY
        """,
        max_rows=20,
    )
    order_options = [(f"{order_id} | {label}", int(order_id)) for order_id, label in order_rows]
    if not order_options:
        order_options = [("1042 | seeded workshop order", 1042)]

    provider_widget = widgets.Dropdown(
        options=[(item["label"], key) for key, item in PROVIDER_BUNDLES.items()],
        value=PROVIDER_KEY,
        description="Provider:",
    )
    order_widget = widgets.Dropdown(
        options=order_options,
        value=1042 if 1042 in {value for _, value in order_options} else order_options[0][1],
        description="Order:",
    )
    question_widget = widgets.Textarea(
        value="Can the customer return this item, and what should support do next?",
        description="Question:",
        layout=widgets.Layout(width="95%", height="90px"),
    )
    submit_widget = widgets.Button(description="Ask Select AI", button_style="primary", icon="search")
    answer_output = widgets.Output()
    diagnostic_output = widgets.Output()
    conversation_by_provider: dict[str, str] = {}


    def handle_support_request(_: widgets.Button) -> None:
        with answer_output:
            clear_output(wait=True)
            selected_provider = str(provider_widget.value).upper()
            selected_order = int(order_widget.value)
            question = question_widget.value.strip()

            if selected_provider not in PROVIDER_BUNDLES:
                print("Choose OCI Generative AI or Ollama.")
                return
            if not question:
                print("Enter a support question.")
                return

            selected_bundle = PROVIDER_BUNDLES[selected_provider]
            request = (
                f"Review order {selected_order}. Answer this question: {question} "
                "Separate verified order facts from policy guidance, cite policy sources, "
                "and identify the next permitted support action."
            )

            try:
                conversation_for_request = conversation_by_provider.get(selected_provider)
                if not conversation_for_request:
                    conversation_for_request = create_conversation(
                        f"order-support-{selected_provider.lower()}"
                    )
                    conversation_by_provider[selected_provider] = conversation_for_request

                answer = run_team(
                    selected_bundle["agent_team"],
                    request,
                    conversation_for_request,
                )
                show_text("Answer", answer)
                ok("Support request completed successfully.")

                with diagnostic_output:
                    clear_output(wait=True)
                    show_table(
                        ["DIAGNOSTIC", "VALUE"],
                        [
                            ("Provider", selected_bundle["label"]),
                            ("NL2SQL profile", selected_bundle["nl2sql_profile"]),
                            ("RAG profile", selected_bundle["rag_profile"]),
                            ("Agent team", selected_bundle["agent_team"]),
                            ("Conversation ID", conversation_for_request),
                        ],
                    )
            except oracledb.DatabaseError as exc:
                error("Select AI could not complete the request: " + short_error(exc))
                print("Check the selected provider health, then retry or choose the other provider.")
            except Exception as exc:
                error("The request could not be completed: " + short_error(exc))


    submit_widget.on_click(handle_support_request)


In [ ]:
with cell_status('Order-support application launched.'):
    # Launch the completed prototype after running either your implementation or the solution.
    display(
        widgets.VBox(
            [
                widgets.HTML("<h3>Retail order-support assistant</h3>"),
                widgets.HBox([provider_widget, order_widget]),
                question_widget,
                submit_widget,
                widgets.HTML("<h4>Answer</h4>"),
                answer_output,
                widgets.HTML("<h4>Diagnostics</h4>"),
                diagnostic_output,
            ]
        )
    )


## Wrap-up

You moved from prompt to inspected SQL, chose the correct action contract, curated or reviewed Feedback, preserved conversation state explicitly, generated disposable data, grounded a policy answer with in-database embeddings, and called a Select AI agent from Python.

Common recovery checks:

- **A provider call times out:** confirm that the selected provider is healthy and, for Ollama, already warm. The database reaches Ollama by its service DNS name, not `localhost`.
- **Generated SQL uses the wrong business meaning:** improve comments and constraints, narrow the object list, inspect with `SHOWSQL`, and add trusted Feedback only in an isolated profile.
- **A second run produces extra prototype rows:** rerun the synthetic-data solution, which resets only `POC_SUPPORT_CASES`.
- **A solution is collapsed:** use the cell gutter or JupyterLab Cell Inspector to expose the cell tagged `solution`.

Optional extension after the timed lab: inspect whether the provider-specific agent team includes the read-only `GET_ALLOWED_SUPPORT_ACTIONS` tool, then compare that deterministic tool result with the agent's recommended action.

### Official references

- [Select AI interfaces](https://docs.oracle.com/en/database/oracle/oracle-database/26/selai/select-ai-interfaces.html)
- [Select AI Feedback](https://docs.oracle.com/en/database/oracle/oracle-database/26/selai/feedback.html)
- [Select AI conversations](https://docs.oracle.com/en/database/oracle/oracle-database/26/selai/select-ai-enable-conversations.html)
- [Select AI Agent framework](https://docs.oracle.com/en/database/oracle/oracle-database/26/selai/select-ai-agent1.html)


In [ ]:
with cell_status('Database connection cleanup completed.'):
    # Close the connection when you are finished with the lab.
    # Re-run the connection/helper cell before reusing the notebook after this point.
    if conn and not conn.is_healthy():
        print("The database connection is already closed or unhealthy.")
    else:
        conn.close()
        print("Database connection closed. Lab complete.")
